# 🔬 ForgeSight AI: Industrial Visual Anomaly Detection (Phase 1)
Welcome to **ForgeSight AI**. This notebook executes the Phase 1 industrial anomaly detection and defect localization pipeline on the **MVTec AD** dataset using **PatchCore** (Pre-trained PyTorch Feature Extractor + Scikit-learn Nearest-Neighbor Memory Bank).

---

### 📋 Pipeline Steps
1. **Workspace Setup & GPU Check**: Verify hardware acceleration (NVIDIA T4 / V100 / A100).
2. **Dataset Acquisition**: Download and extract the target MVTec AD category (`bottle`).
3. **Memory Bank Fitting**: Extract patch features from normal training images and index with Scikit-learn.
4. **Evaluation**: Benchmark Image-level ROC-AUC, Pixel-level ROC-AUC, and Per-Region Overlap (AU-PRO).
5. **Visual Inspection**: Display 4-panel diagnostic overlays (Input | Ground Truth | Heatmap | Defect Contour).
6. **Export Artifacts**: Download checkpoints, metric reports, and inspection figures.

## 🛠️ Step 1: Workspace Setup

Extracts `ForgeSight_AI_colab.zip` and sets the working directory.

In [ ]:
import os
from pathlib import Path

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if Path("/content/ForgeSight_AI_colab.zip").exists():
        !unzip -q -o /content/ForgeSight_AI_colab.zip -d /content/ForgeSight_AI
        %cd /content/ForgeSight_AI
    elif Path("/content/ForgeSight_AI").exists():
        %cd /content/ForgeSight_AI
    else:
        print("[*] Tip: Upload 'ForgeSight_AI_colab.zip' to Colab's file panel, or mount Google Drive.")

print(f"Current working directory: {os.getcwd()}")
assert Path("src").exists(), "Error: 'src' folder not found. Please ensure you are in the ForgeSight_AI directory."

## ⚡ Step 2: GPU Check & Dependencies
Verify GPU acceleration and install required packages.

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("[!] Note: Running on CPU. For faster processing, go to Runtime -> Change runtime type -> T4 GPU.")

# Install dependencies
!pip install -q -r requirements.txt

## 📦 Step 3: Download MVTec AD Category (`bottle`)

Downloads `bottle` directly into Colab storage (~355 images across normal training, defect test, and ground-truth masks).

In [ ]:
!python data/download_mvtec.py --category bottle

## 🧠 Step 4: Fit Memory Bank (Normal Feature Distribution)

1. **Pre-trained CNN Backbone (`resnet18`):** Processes normal training images.
2. **Multi-Scale Feature Hooks:** Captures representations from `layer2` (texture) and `layer3` (structure).
3. **Patch Context Pooling:** Incorporates spatial neighborhood context.
4. **Coreset Subsampling ($10\%$):** Compresses the patch memory bank for rapid distance queries.
5. **Scikit-learn Nearest Neighbors:** Builds an index for Euclidean distance queries.

In [ ]:
!python scripts/run_train.py --category bottle

## 📊 Step 5: Test Set Evaluation & Benchmark Metrics

Calculates real benchmark metrics on the test set:
- **Image-level ROC-AUC & F1-score:** Detects whether a bottle is defective.
- **Pixel-level ROC-AUC:** Measures defect segmentation accuracy across all pixels.
- **Per-Region Overlap (AU-PRO):** Industrial benchmark treating every defect region equally up to $30\%$ FPR.

In [ ]:
!python scripts/run_evaluate.py --category bottle --num_vis 10

## 🔍 Step 6: Diagnostic Visual Inspection (Heatmap Overlays)

Displays 4-panel diagnostic panels inline:
- **Input Sample:** The test image.
- **Ground Truth Mask:** Human-annotated defect boundary.
- **Predicted Anomaly Map:** Continuous patch distance score heatmap.
- **Overlay:** Continuous heatmap blended over the image with predicted defect contour in red.

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

heatmap_dir = Path("outputs/heatmaps/bottle")
if heatmap_dir.exists():
    inspection_panels = sorted(list(heatmap_dir.glob("*.png")))
    print(f"[+] Displaying {len(inspection_panels)} exported diagnostic panels:\n")
    for panel_path in inspection_panels:
        img = Image.open(panel_path)
        plt.figure(figsize=(16, 4))
        plt.imshow(img)
        plt.axis("off")
        plt.show()
else:
    print("[!] No heatmaps found. Please run Step 5 first.")

## 💾 Step 7: Export Benchmark Results and Visualizations

Download the evaluation report, model checkpoint, and heatmaps as a single `.zip` file.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

if Path("outputs").exists():
    shutil.make_archive("bottle_results", "zip", "outputs")
    print("[+] Results packaged as 'bottle_results.zip'. Downloading...")
    files.download("bottle_results.zip")
else:
    print("[!] 'outputs' directory does not exist yet. Please run Steps 4 and 5 first.")